In [ ]:
!pip -q install -U transformers torchaudio jiwer soundfile accelerate pandas

import zipfile, glob
from pathlib import Path

# audios.zip -> audios/
if Path("audios.zip").exists():
    with zipfile.ZipFile("audios.zip", "r") as z:
        z.extractall("audios")

# train/val/test zip -> splits/
for name in ["train.zip","val.zip","test.zip"]:
    if Path(name).exists():
        with zipfile.ZipFile(name, "r") as z:
            z.extractall("splits")

print("Audio files:", len(glob.glob("audios/**/*.ogg", recursive=True)))
print("Split files sample:", glob.glob("splits/**/*.*", recursive=True)[:15])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 133.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 126.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas

In [ ]:
import pandas as pd, os, re

def fix_paths(csv_in, csv_out):
    df = pd.read_csv(csv_in)
    df["audio"] = df["audio"].str.replace(r"^audios/", "audios/audios/", regex=True)
    df.to_csv(csv_out, index=False, encoding="utf-8-sig")
    print("saved:", csv_out)

fix_paths("splits/train/train.csv", "splits/train/train_fixed.csv")
fix_paths("splits/val/val.csv",     "splits/val/val_fixed.csv")
fix_paths("splits/test/test.csv",   "splits/test/test_fixed.csv")

t = pd.read_csv("splits/train/train_fixed.csv")
print("example:", t.iloc[0]["audio"])
print("exists?", os.path.exists(t.iloc[0]["audio"]))

saved: splits/train/train_fixed.csv
saved: splits/val/val_fixed.csv
saved: splits/test/test_fixed.csv
example: audios/audios/Voice/32.ogg
exists? True


In [ ]:
import re

AR_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652]")
TATWEEL = re.compile(r"\u0640")
EXTRA_SPACES = re.compile(r"\s+")
PUNCT_SPACES = re.compile(r"\s*([،؛؟,.!])\s*")

AR_INDIC_TO_LATIN = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
FA_INDIC_TO_LATIN = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")

# توحيد بعض الكلمات الشائعة
WORD_MAP_STRONG = {
    "هاذا": "هذا",
    "هده": "هذه",
    "هدي": "هذه",
    "علشان": "عشان",
    "عشان": "من اجل",
    "الدواء": "دواء",
    "الدوا": "دواء",
    "الدول": "دواء",
    "حبه": "حبة",
    "حبات": "حبات",
    "كبسوله": "كبسولة",
    "معلقة": "ملعقة",
    "معلقه": "ملعقة",
    "نقطه": "نقطة",
    "تحميله": "تحميلة",
    "لبوسه": "لبوس",
}

NUM_WORDS = {
    "واحد":"1","واحدة":"1",
    "اثنين":"2","اتنين":"2",
    "ثلاث":"3","تلات":"3",
    "اربعة":"4","أربعة":"4",
    "خمسة":"5","ستة":"6","سبعة":"7",
    "ثمانية":"8","تسعة":"9","عشرة":"10"
}

PHRASE_RULES = [
    # timing
    (r"\bقبل\s+الاكل\b", "قبل الاكل"),
    (r"\bبعد\s+الاكل\b", "بعد الاكل"),
    (r"\b(مع|اثناء)\s+الاكل\b", "مع الاكل"),
    (r"\bعالريق\b", "على الريق"),
    (r"\bعلي\s+الريق\b", "على الريق"),
    (r"\bعلى\s+الريق\b", "على الريق"),
    (r"\bقبل\s+الفطور\b", "قبل الفطور"),
    (r"\bبعد\s+الفطور\b", "بعد الفطور"),
    (r"\bقبل\s+النوم\b", "قبل النوم"),

    # frequency
    (r"\bمرة\s+باليوم\b", "مرة يوميا"),
    (r"\bمرتين\s+باليوم\b", "مرتين يوميا"),
    (r"\b(ثلاث|تلات)\s+مرات\s+باليوم\b", "3 مرات يوميا"),
    (r"\bكل\s*4\s*ساع(?:ة|ات)\b", "كل 4 ساعات"),
    (r"\bكل\s*6\s*ساع(?:ة|ات)\b", "كل 6 ساعات"),
    (r"\bكل\s*8\s*ساع(?:ة|ات)\b", "كل 8 ساعات"),
    (r"\bكل\s*12\s*ساع(?:ة|ات)\b", "كل 12 ساعة"),
    (r"\b(وقت|عند)\s+الل\w+وم\b", "عند اللزوم"),

    # duration
    (r"\b(ثلاث|تلات)\s+ايام\b", "3 ايام"),
    (r"\bعشرة\s+ايام\b", "10 ايام"),
    (r"\bاسبوعين\b", "2 اسبوع"),
    (r"\bشهرين\b", "2 شهر"),
]

def normalize_text_base(text: str) -> str:
    text = str(text or "").strip()

    # إزالة التشكيل والتطويل
    text = AR_DIACRITICS.sub("", text)
    text = TATWEEL.sub("", text)

    # توحيد الألف والهمزات والياء
    text = re.sub(r"[أإآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و").replace("ئ", "ي")

    # أرقام عربية -> لاتينية
    text = text.translate(AR_INDIC_TO_LATIN).translate(FA_INDIC_TO_LATIN)

    # 2,5 -> 2.5
    text = re.sub(r"(\d),(\d)", r"\1.\2", text)

    # مسافات حول الترقيم
    text = PUNCT_SPACES.sub(r" \1 ", text)
    text = EXTRA_SPACES.sub(" ", text).strip()

    return text


def normalize_text_strong(text: str) -> str:
    text = normalize_text_base(text)

    # كلمات أرقام -> أرقام
    keys = sorted(NUM_WORDS.keys(), key=len, reverse=True)
    pat = r"\b(" + "|".join(map(re.escape, keys)) + r")\b"
    text = re.sub(pat, lambda m: NUM_WORDS[m.group(0)], text)

    # قواعد العبارات
    for pat, rep in PHRASE_RULES:
        text = re.sub(pat, rep, text)

    # توحيد الكلمات
    keys = sorted(WORD_MAP_STRONG.keys(), key=len, reverse=True)
    pat = r"\b(" + "|".join(map(re.escape, keys)) + r")\b"
    text = re.sub(pat, lambda m: WORD_MAP_STRONG[m.group(0)], text)

    # مسافات نهائية
    text = EXTRA_SPACES.sub(" ", text).strip()
    return text

In [ ]:
import torch, torchaudio, soundfile as sf
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-large-v3"

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="arabic", task="transcribe")

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model = model.to("cuda", dtype=torch.float16)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="arabic", task="transcribe")

print("Model loaded FP16 on GPU.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model loaded FP16 on GPU.


In [ ]:
import numpy as np
import torch.nn.functional as F

def load_audio_16k(path):
    audio, sr = sf.read(path)
    if len(audio.shape) > 1:
        audio = audio.mean(axis=1)
    audio = torch.tensor(audio, dtype=torch.float32)
    if sr != 16000:
        audio = torchaudio.functional.resample(audio, sr, 16000)
    return audio


def apply_vad_chunks(waveform, sample_rate=16000, frame_ms=30, energy_threshold=0.0005):
    frame_len = int(sample_rate * frame_ms / 1000)
    if frame_len <= 0:
        return [waveform.numpy()]

    chunks = []
    current = []

    for i in range(0, len(waveform), frame_len):
        frame = waveform[i:i+frame_len]
        if len(frame) < frame_len:
            break
        energy = (frame**2).mean().item()
        if energy > energy_threshold:
            current.append(frame)
        else:
            if current:
                chunks.append(torch.cat(current))
                current = []
    if current:
        chunks.append(torch.cat(current))

    if not chunks:
        return [waveform.numpy()]
    return [c.numpy() for c in chunks]


def transcribe_batch(paths, use_vad=True):
    all_audios = []

    # 1) Load + VAD
    for p in paths:
        wf = load_audio_16k(p)
        if use_vad:
            chunks = apply_vad_chunks(wf, sample_rate=16000)
            if len(chunks) > 1:
                sep = torch.zeros(int(0.2 * 16000))
                merged = []
                for c in chunks:
                    merged.append(torch.tensor(c))
                    merged.append(sep)
                wf_merged = torch.cat(merged)
                all_audios.append(wf_merged.numpy())
            else:
                all_audios.append(chunks[0])
        else:
            all_audios.append(wf.numpy())

    # 2) Processor → mel features
    inputs = processor(all_audios, sampling_rate=16000, return_tensors="pt", padding=True)
    mel = inputs.input_features  # shape: (batch, 80, T)

    # 3) PAD TO 3000 FRAMES (حل المشكلة)
    max_len = 3000
    T = mel.shape[-1]

    if T < max_len:
        pad_len = max_len - T
        mel = F.pad(mel, (0, pad_len), mode="constant", value=0.0)
    elif T > max_len:
        mel = mel[:, :, :max_len]  # قص لو أطول من 3000

    mel = mel.to("cuda", dtype=torch.float16)

    # 4) Generate
    with torch.no_grad():
        pred_ids = model.generate(
            mel,
            temperature=0,
            num_beams=5
        )

    # 5) Decode + strong normalization
    texts = processor.batch_decode(pred_ids, skip_special_tokens=True)
    return [normalize_text_strong(t) for t in texts]

In [ ]:
import pandas as pd

test_df = pd.read_csv("splits/test/test_fixed.csv")
sample = test_df.sample(n=min(5, len(test_df)), random_state=42)

hyps = transcribe_batch(sample["audio"].tolist(), use_vad=True)

for i, (idx, row) in enumerate(sample.iterrows()):
    ref = normalize_text_strong(row.text)
    hyp = hyps[i]
    print("\n---", i+1, "---")
    print("AUDIO:", row.audio)
    print("REF:", ref)
    print("HYP:", hyp)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA


--- 1 ---
AUDIO: audios/audios/Voice/37.ogg
REF: هاد بخاخ للانف لعلاج حساسية الربيع . استخدم بختين بكل فتحة انف مرة وحدة باليوم الصبح . قبل ما تستخدمه ، نظف انفك منيح وخض العلبة . مفعوله بياخد كم يوم ليبين بشكل كامل ، لهيك لازم تستمر عليه يوميا خلال موسم الحساسية حتي لو ما حسيت باعراض .
HYP: بخض الانف لعلاج حساسية تربية . استخدم بخضين في كل فتحة انف مرة 1 في اليوم الصدر . قبل ما تستخدمه نضف انفك منيح بفضل حلبة . مفروض بيحفظ كم يوم ليبين التكسل الكامل . لهيك لازم تستمر علي يوميا في الموسم الحساسية حتي لو ما حسيت باعراض .

--- 2 ---
AUDIO: audios/audios/Voice/41.ogg
REF: هاد دواء لهرمون الغدة الدرقية . الجرعة هي حبة وحدة كل يوم الصبح علي معدة فاضية تماما ، قبل الفطور بنص ساعة لساعة . لازم تبلعها مع كاسة مي كاملة وبس . لا تاخذ معها اي فيتامينات او مكملات حديد بنفس الوقت ، خلي في فرق 4 ساعات علي الاقل .
HYP: هذا الدوالة هرمون الودة الدردية الجرعة هي حبة 1 كل يوم الصبح علي معدة فاضية تماما قبل الفطور بنصة علي الساعة لازم تبلع مع كاسة مي كاملة وبس لاتاخد معها اي فيتامينات او مكملات عزيزة بن

In [ ]:
from jiwer import wer, cer
import time

train_df = pd.read_csv("splits/train/train_fixed.csv")
val_df   = pd.read_csv("splits/val/val_fixed.csv")
test_df  = pd.read_csv("splits/test/test_fixed.csv")

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print("Total files:", len(all_df))

results = []
refs_all, hyps_all = [], []
start = time.time()

BATCH = 4

for i in range(0, len(all_df), BATCH):
    batch = all_df.iloc[i:i+BATCH]
    paths = batch["audio"].tolist()

    try:
        hyps = transcribe_batch(paths, use_vad=True)
    except Exception as e:
        print("Error in batch", i, ":", e)
        hyps = [""] * len(batch)

    for j, (idx, row) in enumerate(batch.iterrows()):
        ref = normalize_text_strong(row["text"])
        hyp = hyps[j]

        refs_all.append(ref)
        hyps_all.append(hyp)

        results.append({
            "id": row.get("id", idx),
            "audio": row["audio"],
            "ref": ref,
            "hyp": hyp,
            "wer": wer([ref],[hyp]),
            "cer": cer([ref],[hyp])
        })

    if (i+1) % 20 == 0:
        print(f"{i+1}/{len(all_df)} | avg WER so far: {wer(refs_all, hyps_all):.4f}")

print("\nFINAL AVG WER:", wer(refs_all, hyps_all))
print("FINAL AVG CER:", cer(refs_all, hyps_all))
print("Minutes:", (time.time()-start)/60)

out_csv = "all_files_largev3_vad_norm_strong_predictions.csv"
pd.DataFrame(results).to_csv(out_csv, index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd

df = pd.read_csv("all_files_largev3_vad_norm_strong_predictions.csv")

worst = df.sort_values("wer", ascending=False).head(20)
worst[["id","audio","wer","cer","ref","hyp"]]

In [ ]:
df = pd.read_csv("all_files_largev3_vad_norm_strong_predictions.csv")
worst = df.sort_values("wer", ascending=False).head(10).copy()

worst["ref_len"] = worst["ref"].astype(str).str.split().str.len()
worst["hyp_len"] = worst["hyp"].astype(str).str.split().str.len()

worst[["id","wer","cer","ref_len","hyp_len","ref","hyp"]]